In [1]:
import boto3
import pandas as pd
from io import StringIO, BytesIO
import os
from dotenv import load_dotenv
import numpy as np
import mlflow
from datetime import datetime
from sentence_transformers import SentenceTransformer
import torch
import joblib

load_dotenv()

# MLflow Configuration
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Resume_Job_Matcher")

c:\Users\Saadan\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1762041252390, experiment_id='1', last_update_time=1762041252390, lifecycle_stage='active', name='Resume_Job_Matcher', tags={}>

In [2]:
# AWS credentials and bucket info
bucket_name = 'resume-matcher-bucket-sahil' # <-- Change to your S3 bucket name
resume_key = 'raw-data/Resume.csv'
job_desc_key = 'raw-data/job_title_des.csv'

# Create an S3 client
s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
)

# Fetch and read Resume.csv
resume_obj = s3.get_object(Bucket=bucket_name, Key=resume_key)
df_resumes = pd.read_csv(StringIO(resume_obj['Body'].read().decode('utf-8')))

# Fetch and read job_title_des.csv
job_desc_obj = s3.get_object(Bucket=bucket_name, Key=job_desc_key)
df_job_description = pd.read_csv(StringIO(job_desc_obj['Body'].read().decode('utf-8')))

print("Resumes loaded:", len(df_resumes))
print("Job Descriptions loaded:", len(df_job_description))

Resumes loaded: 2484
Job Descriptions loaded: 2277


In [3]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(model_name, device=device)

run_name = f"SBERT_Embedding_Generation_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

with mlflow.start_run(run_name=run_name) as run:
    print(f"Starting MLflow run: {run.info.run_name}")
    mlflow.log_param("sentence_transformer_model", model_name)
    mlflow.log_param("device", device)
    mlflow.log_param("num_job_descriptions", len(df_job_description))

    # Generate embeddings for job descriptions
    job_descriptions_text = df_job_description["Job Description"].tolist()
    job_embeddings = model.encode(job_descriptions_text, convert_to_tensor=True, show_progress_bar=True)
    
    print("Embeddings generated with shape:", job_embeddings.shape)
    mlflow.log_metric("embedding_dimension", job_embeddings.shape[1])

    # --- Save model and embeddings as artifacts ---
    # 1. Save the SentenceTransformer model
    model_path = "sbert_model"
    model.save(model_path)
    mlflow.log_artifact(model_path, artifact_path="sbert_model_artifact")
    
    # 2. Save the pre-computed job embeddings tensor
    embeddings_path = "job_embeddings.pt"
    torch.save(job_embeddings, embeddings_path)
    mlflow.log_artifact(embeddings_path, artifact_path="embeddings_artifact")

    # Register the model in MLflow Model Registry
    mlflow.pyfunc.log_model(
        artifact_path="sbert_pyfunc",
        python_model=mlflow.pyfunc.PythonModel(), # Using a dummy model for registration purposes
        registered_model_name="Resume_Matcher_SBERT"
    )
    
    # --- Upload artifacts to S3 directly ---
    # Upload the saved model folder to S3
    s3.upload_file(embeddings_path, bucket_name, f'models/{embeddings_path}')
    print(f"Embeddings uploaded to s3://{bucket_name}/models/{embeddings_path}")
    
    # For the model, you would typically zip and upload or upload file by file
    # For simplicity, we assume the model from MLflow artifacts is sufficient for deployment.

print("MLflow run completed.")

Starting MLflow run: SBERT_Embedding_Generation_20251103_030830


Batches: 100%|██████████| 72/72 [00:48<00:00,  1.47it/s]


Embeddings generated with shape: torch.Size([2277, 384])


2025/11/03 03:09:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/03 03:09:34 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Resume_Matcher_SBERT' already exists. Creating a new version of this model...
2025/11/03 03:09:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Resume_Matcher_SBERT, version 2
Created version '2' of model 'Resume_Matcher_SBERT'.


Embeddings uploaded to s3://resume-matcher-bucket-sahil/models/job_embeddings.pt
🏃 View run SBERT_Embedding_Generation_20251103_030830 at: http://localhost:5000/#/experiments/1/runs/9b4ea9cbc8414413a9b6871de01a95b2
🧪 View experiment at: http://localhost:5000/#/experiments/1
MLflow run completed.
